# NLP + Finance: Modeling & Evaluation

This notebook trains and evaluates sentiment classifiers on the Financial PhraseBank dataset. Preprocessing is handled by `src/preprocess.py`.

**Models:** Logistic Regression, Naive Bayes, LinearSVC  
**Baseline:** VADER (zero-shot, no training)  
**Metric:** Weighted F1 — chosen due to class imbalance (neutral 59.4%, negative 12.5%)

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.preprocess import preprocess

# Load data
df = pd.read_csv('../data/raw/all-data.csv',
                 encoding='latin-1',
                 header=None,
                 names=['sentiment', 'headline'])

# Apply preprocessing
df['cleaned'] = df['headline'].apply(preprocess)

print(f"Dataset shape: {df.shape}")
print(f"\nSample:")
print(df[['sentiment', 'headline', 'cleaned']].head(3))

Dataset shape: (4846, 3)

Sample:
  sentiment                                           headline  \
0   neutral  According to Gran , the company has no plans t...   
1   neutral  Technopolis plans to develop in stages an area...   
2  negative  The international electronic industry company ...   

                                             cleaned  
0  according gran company plan move production ru...  
1  technopolis plan develop stage area less squar...  
2  international electronic industry company elco...  


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

X = df['cleaned']
y = df['sentiment']

# Train/test split — 80/20, stratified to preserve class ratios
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF vectorization
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"Train size: {X_train_tfidf.shape}")
print(f"Test size:  {X_test_tfidf.shape}")
print(f"\nClass distribution in test set:")
print(y_test.value_counts())

Train size: (3876, 5000)
Test size:  (970, 5000)

Class distribution in test set:
sentiment
neutral     576
positive    273
negative    121
Name: count, dtype: int64


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, f1_score

models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Naive Bayes': MultinomialNB(),
    'LinearSVC': LinearSVC(class_weight='balanced', random_state=42, max_iter=1000)
}

results = {}

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    f1 = f1_score(y_test, y_pred, average='weighted')
    results[name] = f1
    print(f"\n{name}")
    print(f"Weighted F1: {f1:.4f}")
    print(classification_report(y_test, y_pred))


Logistic Regression
Weighted F1: 0.7269
              precision    recall  f1-score   support

    negative       0.55      0.69      0.61       121
     neutral       0.81      0.80      0.80       576
    positive       0.64      0.59      0.62       273

    accuracy                           0.73       970
   macro avg       0.67      0.69      0.68       970
weighted avg       0.73      0.73      0.73       970


Naive Bayes
Weighted F1: 0.6662
              precision    recall  f1-score   support

    negative       0.81      0.21      0.34       121
     neutral       0.71      0.96      0.82       576
    positive       0.67      0.40      0.50       273

    accuracy                           0.71       970
   macro avg       0.73      0.52      0.55       970
weighted avg       0.71      0.71      0.67       970


LinearSVC
Weighted F1: 0.7307
              precision    recall  f1-score   support

    negative       0.56      0.62      0.59       121
     neutral       0.79 

LinearSVC and Logistic Regression are basically tied at ~0.73 weighted F1. Naive Bayes is noticeably worse, and the per-class numbers show exactly why. its recall on negative is 0.21, meaning it's missing 80% of negative headlines. that's the worst possible failure mode for a trading system. class_weight='balanced' is clearly doing real work for the other two models - without it the results would probably look a lot more like Naive Bayes.